# 03. Análisis Exploratorio de Datos (EDA)

- **Autor:** Jhonner Acosta
- **Fecha:** 2026-08-20
- **Descripción:** Análisis exploratorio completo del dataset de enfermedades cardíacas: análisis univariable, bivariable y multivariable. Identificación de transformaciones necesarias, relaciones entre variables y viabilidad de un modelo heurístico.

## 0. Imports y configuración

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

_root = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists())
DATA_DIR = _root / "data"

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 72
plt.rcParams["figure.figsize"] = (10, 5)

SEED = 42
rng = np.random.default_rng(SEED)

## 1. Carga de datos

In [2]:
df = pd.read_parquet(DATA_DIR / "02_intermediate/corazon_type_fixed.parquet")

# Columna auxiliar disease como string para compatibilidad con seaborn
df["disease_str"] = df["disease"].astype(str)  # "True" / "False"

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.drop(columns=["disease_str"]).sample(5, random_state=SEED)

Filas: 2864 | Columnas: 15


,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
1590,41,Female,nontypical,126,306,False,normal,163,<NA>,0.0,1,0,normal,False
1752,45,Male,asymptomatic,115,260,False,left ventricular hypertrophy,185,<NA>,0.0,1,0,normal,False
772,48,Male,asymptomatic,124,274,False,left ventricular hypertrophy,166,<NA>,0.5,2,0,reversable,True
1735,68,Male,nonanginal,180,274,True,left ventricular hypertrophy,150,<NA>,1.6,2,0,reversable,True
387,52,Male,nontypical,120,325,False,normal,172,<NA>,0.2,1,0,normal,False


## 2. Descripción del esquema de datos

### 2.1 Schema y tipos

In [3]:
df.drop(columns=["disease_str"]).info()

<class 'pandas.DataFrame'>
RangeIndex: 2864 entries, 0 to 2863
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         2864 non-null   int8   
 1   sex         2818 non-null   str    
 2   chest_pain  2817 non-null   str    
 3   rest_bp     2864 non-null   int16  
 4   chol        2864 non-null   int16  
 5   fbs         2818 non-null   boolean
 6   rest_ecg    2818 non-null   str    
 7   max_hr      2864 non-null   int16  
 8   exang       0 non-null      boolean
 9   old_peak    2864 non-null   float32
 10  slope       2864 non-null   int64  
 11  ca          2824 non-null   Int8   
 12  thal        2842 non-null   str    
 13  disease     2864 non-null   boolean
dtypes: Int8(1), boolean(3), float32(1), int16(3), int64(1), int8(1), str(4)
memory usage: 276.3 KB


In [4]:
schema = pd.DataFrame({
    "dtype": df.drop(columns=["disease_str"]).dtypes,
    "nulos": df.drop(columns=["disease_str"]).isnull().sum(),
    "pct_nulos": (df.drop(columns=["disease_str"]).isnull().sum() / len(df) * 100).round(2),
    "únicos": df.drop(columns=["disease_str"]).nunique(),
    "tipo_variable": [
        "Numérica continua",   # age
        "Categórica nominal",  # sex
        "Categórica nominal",  # chest_pain
        "Numérica continua",   # rest_bp
        "Numérica continua",   # chol
        "Booleana",            # fbs
        "Categórica nominal",  # rest_ecg
        "Numérica continua",   # max_hr
        "Booleana",            # exang
        "Numérica continua",   # old_peak
        "Categórica ordinal",  # slope
        "Numérica discreta",   # ca
        "Categórica nominal",  # thal
        "Booleana (target)",   # disease
    ],
    "util_ml": [
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "Sí", "Sí", "Sí", "Sí",
        "Sí", "Sí", "Sí", "Target",
    ],
})
schema

,dtype,nulos,pct_nulos,únicos,tipo_variable,util_ml
age,int8,0,0.00,41,Numérica continua,Sí
sex,str,46,1.61,2,Categórica nominal,Sí
chest_pain,str,47,1.64,4,Categórica nominal,Sí
rest_bp,int16,0,0.00,50,Numérica continua,Sí
chol,int16,0,0.00,152,Numérica continua,Sí
fbs,boolean,46,1.61,2,Booleana,Sí
rest_ecg,str,46,1.61,3,Categórica nominal,Sí
max_hr,int16,0,0.00,91,Numérica continua,Sí
exang,boolean,2864,100.00,0,Booleana,Sí
old_peak,float32,0,0.00,40,Numérica continua,Sí


### 2.2 Duplicados

In [5]:
df_orig = df.drop(columns=["disease_str"])
n_dup_filas = df_orig.duplicated().sum()
n_dup_cols = df_orig.T.duplicated().sum()
print(f"Filas exactamente duplicadas: {n_dup_filas}")
print(f"Columnas exactamente duplicadas: {n_dup_cols}")
if n_dup_filas > 0:
    print("\nFilas duplicadas:")
    display(df_orig[df_orig.duplicated(keep=False)].sort_values(list(df_orig.columns)))

Filas exactamente duplicadas: 2460
Columnas exactamente duplicadas: 0

Filas duplicadas:


,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
132,29,Male,nontypical,130,204,False,left ventricular hypertrophy,202,<NA>,0.0,1,0,normal,False
435,29,Male,nontypical,130,204,False,left ventricular hypertrophy,202,<NA>,0.0,1,0,normal,False
726,29,Male,nontypical,130,204,False,left ventricular hypertrophy,202,<NA>,0.0,1,0,normal,False
1000,29,Male,nontypical,130,204,False,left ventricular hypertrophy,202,<NA>,0.0,1,0,normal,False
1215,29,Male,nontypical,130,204,False,left ventricular hypertrophy,202,<NA>,0.0,1,0,normal,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1510,77,Male,asymptomatic,125,304,False,left ventricular hypertrophy,162,<NA>,0.0,1,3,normal,True
1813,77,Male,asymptomatic,125,304,False,left ventricular hypertrophy,162,<NA>,0.0,1,3,normal,True
2116,77,Male,asymptomatic,125,304,False,left ventricular hypertrophy,162,<NA>,0.0,1,3,normal,True
2419,77,Male,asymptomatic,125,304,False,left ventricular hypertrophy,162,<NA>,0.0,1,3,normal,True


### 2.3 Balance del target

In [6]:
# Conteo directo evitando indexación por bool en dtype nullable
n_positivo = int(df["disease"].astype(bool).sum())
n_negativo = len(df) - n_positivo

print("Distribución del target (disease):")
print(f"True  (Con enfermedad):  {n_positivo} ({n_positivo/len(df):.1%})")
print(f"False (Sin enfermedad):  {n_negativo} ({n_negativo/len(df):.1%})")
print(f"\nRatio positivo/total: {n_positivo/len(df):.2%}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Sin enfermedad (False)", "Con enfermedad (True)"],
       [n_negativo, n_positivo],
       color=["steelblue", "tomato"])
ax.set_title("Balance del target: disease")
ax.set_ylabel("Frecuencia")
for i, v in enumerate([n_negativo, n_positivo]):
    ax.text(i, v + 5, f"{v} ({v/len(df):.1%})", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

Distribución del target (disease):
True  (Con enfermedad):  1295 (45.2%)
False (Sin enfermedad):  1569 (54.8%)

Ratio positivo/total: 45.22%


/tmp/ipykernel_932/3011004599.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Análisis Univariable

### 3.1 Variables numéricas — Estadística descriptiva

In [7]:
num_cols = ["age", "rest_bp", "chol", "max_hr", "old_peak"]

stats_df = df[num_cols].agg([
    "count", "mean", "median", lambda x: x.mode()[0],
    "std", "var", "min", "max"
]).T
stats_df.columns = ["count", "mean", "median", "mode", "std", "var", "min", "max"]

stats_df["range"] = stats_df["max"] - stats_df["min"]
stats_df["IQR"] = df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25)
stats_df["Q1"] = df[num_cols].quantile(0.25)
stats_df["Q3"] = df[num_cols].quantile(0.75)
stats_df["skewness"] = df[num_cols].skew().values
stats_df["kurtosis"] = df[num_cols].kurt().values

stats_df.round(3)

,count,mean,median,mode,std,var,min,max,range,IQR,Q1,Q3,skewness,kurtosis
age,2864.0,54.428,55.0,58.0,9.018,81.317,29.0,77.0,48.0,13.00,48.00,61.0,-0.211,-0.524
rest_bp,2864.0,131.706,130.0,130.0,17.429,303.765,94.0,200.0,106.0,20.00,120.00,140.0,0.724,0.976
chol,2864.0,246.197,240.0,240.0,50.779,2578.546,126.0,564.0,438.0,62.00,212.00,274.0,1.099,4.202
max_hr,2864.0,149.611,153.0,162.0,22.806,520.120,71.0,202.0,131.0,32.25,133.75,166.0,-0.529,-0.066
old_peak,2864.0,1.038,0.8,0.0,1.163,1.351,0.0,6.2,6.2,1.60,0.00,1.6,1.270,1.547


### 3.2 Distribuciones numéricas (histograma + KDE + boxplot)

In [8]:
fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, 4 * len(num_cols)))

for i, col in enumerate(num_cols):
    data = df[col].dropna()

    sns.histplot(data, kde=True, ax=axes[i, 0], color="steelblue")
    axes[i, 0].set_title(f"{col} — Distribución")
    axes[i, 0].set_xlabel(col)

    sns.boxplot(y=data, ax=axes[i, 1], color="lightcoral")
    axes[i, 1].set_title(f"{col} — Boxplot (outliers)")

    sk = data.skew()
    axes[i, 0].annotate(f"skew={sk:.2f}", xy=(0.97, 0.92),
                        xycoords="axes fraction", ha="right", fontsize=9,
                        color="darkred")

plt.suptitle("Variables numéricas — Distribuciones", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/1460188737.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.3 Test de normalidad (Shapiro-Wilk)

In [9]:
normality_results = []
alpha = 0.05
for col in num_cols:
    data = df[col].dropna()
    sample = data.sample(min(3000, len(data)), random_state=SEED)
    stat, p = stats.shapiro(sample)
    normality_results.append({
        "columna": col,
        "stat": round(stat, 4),
        "p_value": round(p, 4),
        "normal (p>0.05)": p > alpha,
    })

pd.DataFrame(normality_results)

,columna,stat,p_value,normal (p>0.05)
0,age,0.9864,0.0,False
1,rest_bp,0.9643,0.0,False
2,chol,0.9482,0.0,False
3,max_hr,0.9765,0.0,False
4,old_peak,0.8421,0.0,False


### 3.4 Detección de outliers — Método IQR

In [10]:
outlier_summary = []
for col in num_cols:
    data = df[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((data < lower) | (data > upper)).sum()
    outlier_summary.append({
        "columna": col,
        "límite_inferior": round(lower, 2),
        "límite_superior": round(upper, 2),
        "n_outliers": n_out,
        "pct_outliers": f"{n_out/len(data)*100:.2f}%",
    })

pd.DataFrame(outlier_summary)

,columna,límite_inferior,límite_superior,n_outliers,pct_outliers
0,age,28.50,80.50,0,0.00%
1,rest_bp,90.00,170.00,84,2.93%
2,chol,119.00,367.00,46,1.61%
3,max_hr,85.38,214.38,9,0.31%
4,old_peak,-2.40,4.00,48,1.68%


### 3.5 Variable numérica discreta: `ca`

In [11]:
ca_counts = df["ca"].value_counts(dropna=False).sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ca_labels = [str(v) if pd.notna(v) else "(sin dato)" for v in ca_counts.index]
sns.barplot(
    x=ca_labels, y=ca_counts.values,
    hue=ca_labels, legend=False,
    ax=ax, color="steelblue"
)
ax.set_title("ca — Número de vasos coloreados (0-3)")
ax.set_xlabel("ca")
ax.set_ylabel("Frecuencia")
for i, v in enumerate(ca_counts.values):
    ax.text(i, v + 1, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/1194941916.py:4: FutureWarning: 

Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:steelblue'` for the same effect.

  sns.barplot(


/tmp/ipykernel_932/1194941916.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.6 Variables categóricas y booleanas

In [12]:
cat_bool_cols = ["sex", "chest_pain", "fbs", "rest_ecg", "exang", "slope", "thal"]
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(cat_bool_cols):
    counts = df[col].value_counts(dropna=False)
    x_labels = [str(v) if pd.notna(v) else "(sin dato)" for v in counts.index]
    sns.barplot(
        x=x_labels, y=counts.values,
        hue=x_labels, legend=False,
        ax=axes[i], palette="muted"
    )
    axes[i].set_title(f"{col}")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Frecuencia")
    axes[i].tick_params(axis="x", rotation=20)
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 1, f"{v}\n({v/len(df):.1%})", ha="center", fontsize=8)
for ax in axes[len(cat_bool_cols):]:
    ax.set_visible(False)
plt.suptitle("Variables categóricas y booleanas — Frecuencias", fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/1498061542.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Análisis Bivariable

### 4.1 Variables numéricas vs. target (`disease`)

In [13]:
PALETTE_D = {"False": "steelblue", "True": "tomato"}

fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 5))

for i, col in enumerate(num_cols):
    sns.boxplot(
        data=df, x="disease_str", y=col, ax=axes[i],
        hue="disease_str",
        order=["False", "True"],
        hue_order=["False", "True"],
        palette=PALETTE_D,
        legend=False
    )
    axes[i].set_title(col)
    axes[i].set_xlabel("disease")

plt.suptitle("Variables numéricas vs. disease (target)", fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/1975529695.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.2 Test estadístico: Mann-Whitney U (numérico vs. target)

In [14]:
disease_bool = df["disease"].astype(bool)

mw_results = []
alpha = 0.05
for col in [*num_cols, "ca"]:
    pos = df.loc[disease_bool, col].dropna()
    neg = df.loc[~disease_bool, col].dropna()
    stat, p = stats.mannwhitneyu(pos, neg, alternative="two-sided")
    mw_results.append({
        "variable": col,
        "U_stat": round(stat, 1),
        "p_value": round(p, 6),
        "significativo (p<0.05)": p < alpha,
        "media_positivo": round(float(pos.mean()), 2),
        "media_negativo": round(float(neg.mean()), 2),
    })

pd.DataFrame(mw_results)

,variable,U_stat,p_value,significativo (p<0.05),media_positivo,media_negativo
0,age,1289382.0,0.0,True,56.59,52.65
1,rest_bp,1159554.5,0.0,True,134.55,129.36
2,chol,1163498.0,0.0,True,251.20,242.06
3,max_hr,514560.0,0.0,True,139.08,158.30
4,old_peak,1491001.0,0.0,True,1.58,0.59
5,ca,1474894.0,0.0,True,1.14,0.29


### 4.3 Variables categóricas vs. target — Chi-cuadrado

In [15]:
cat_cols = ["sex", "chest_pain", "fbs", "rest_ecg", "exang", "slope", "thal"]
chi2_results = []
alpha = 0.05
for col in cat_cols:
    ct = pd.crosstab(
        df[col].astype(object).astype(str),
        df["disease"].astype(object).astype(str),
    )
    if ct.size == 0:
        continue
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    chi2_results.append({
        "variable": col,
        "chi2": round(chi2, 2),
        "p_value": round(p, 6),
        "dof": dof,
        "significativo (p<0.05)": p < alpha,
    })
pd.DataFrame(chi2_results)

,variable,chi2,p_value,dof,significativo (p<0.05)
0,sex,204.86,0.000000,1,True
1,chest_pain,764.03,0.000000,3,True
2,fbs,0.78,0.376477,1,False
3,rest_ecg,95.76,0.000000,2,True
4,slope,447.42,0.000000,2,True
5,thal,784.87,0.000000,2,True


### 4.4 Distribución condicional categóricas vs. target

In [16]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    col_str = df[col].astype(object).fillna("sin dato").astype(str)
    prop = (
        df.assign(col_str=col_str)
        .groupby(["col_str", "disease_str"], observed=True)
        .size()
        .unstack(fill_value=0)
        .apply(lambda r: r / r.sum(), axis=1)
    )
    if prop.empty:
        axes[i].set_title(f"{col} — sin datos")
        continue
    for c in ["False", "True"]:
        if c not in prop.columns:
            prop[c] = 0.0
    prop = prop[["False", "True"]]
    prop.plot(kind="bar", stacked=True, ax=axes[i],
              color=["steelblue", "tomato"], legend=(i == 0))
    axes[i].set_title(f"{col} vs disease")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Proporción")
    axes[i].tick_params(axis="x", rotation=20)
for ax in axes[len(cat_cols):]:
    ax.set_visible(False)
plt.suptitle("Distribución condicional: categóricas vs disease", fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/1452478293.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.5 Correlación entre variables numéricas (Spearman)

In [17]:
df_corr = df[[*num_cols, "ca"]].copy()
df_corr["disease"] = df["disease"].astype(int)

corr_matrix = df_corr.corr(method="spearman", numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax
)
ax.set_title("Correlación de Spearman — variables numéricas + target")
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/1247476256.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.6 Scatter plots clave

In [18]:
pairs = [("age", "max_hr"), ("age", "chol"), ("old_peak", "max_hr")]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (x, y) in zip(axes, pairs, strict=False):
    sns.scatterplot(
        data=df, x=x, y=y,
        hue="disease_str",
        hue_order=["False", "True"],
        palette=PALETTE_D,
        alpha=0.5, ax=ax
    )
    ax.set_title(f"{x} vs {y}")
    ax.legend(title="disease")

plt.suptitle("Scatter plots por disease", fontsize=14)
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/109680449.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Análisis Multivariable

### 5.1 Pairplot — variables más discriminantes

In [19]:
top_vars = ["age", "max_hr", "old_peak", "ca", "disease_str"]
df_pair = df[top_vars].copy()
df_pair["ca"] = df_pair["ca"].astype(float)

g = sns.pairplot(
    df_pair, hue="disease_str",
    hue_order=["False", "True"],
    palette=PALETTE_D,
    diag_kind="kde", plot_kws={"alpha": 0.4}
)
g._legend.set_title("disease")
g.fig.suptitle("Pairplot — variables más discriminantes vs disease",
               y=1.02, fontsize=13)
plt.show()

/tmp/ipykernel_932/3044494951.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.2 Heatmap de correlación completo (encoding de categóricas)

In [20]:
df_enc = df.drop(columns=["disease_str"]).copy()
df_enc["sex"]        = df_enc["sex"].astype("category").cat.codes
df_enc["chest_pain"] = df_enc["chest_pain"].astype("category").cat.codes
df_enc["rest_ecg"]   = df_enc["rest_ecg"].astype("category").cat.codes
df_enc["thal"]       = df_enc["thal"].astype("category").cat.codes
df_enc["slope"]      = df_enc["slope"].astype("category").cat.codes
df_enc["fbs"]        = df_enc["fbs"].astype(float)
df_enc["exang"]      = df_enc["exang"].astype(float)
df_enc["disease"]    = df_enc["disease"].astype(float)
df_enc["ca"]         = df_enc["ca"].astype(float)
corr_all = df_enc.corr(method="spearman", numeric_only=True)
fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_all, dtype=bool))
sns.heatmap(
    corr_all, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.4, annot_kws={"size": 8}, ax=ax
)
ax.set_title("Correlación Spearman — todas las variables (encoded)", fontsize=13)
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/3268230025.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.3 Importancia relativa — Correlación con el target

In [21]:
target_corr = corr_all["disease"].drop("disease").abs().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
target_corr.plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(0.1, color="red", linestyle="--", alpha=0.6, label="umbral 0.1")
ax.set_title("|Correlación Spearman| con disease")
ax.set_xlabel("|rho|")
ax.legend()
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/2322774839.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.4 Heatmap chest_pain × thal (tasa de disease)

In [22]:
pivot = (
    df.assign(disease_num=df["disease"].astype(int))
    .groupby(["chest_pain", "thal"], observed=True)["disease_num"]
    .mean()
    .unstack()
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot.astype(float), annot=True, fmt=".2f",
            cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_title("Tasa de disease por chest_pain x thal")
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/3325110778.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Reglas de validación de datos

Basadas en el análisis univariable, se definen las siguientes reglas de validación que serán usadas en la etapa de ingeniería de datos:

In [23]:
validation_rules = pd.DataFrame([
    {"columna": "age",       "regla": "int",   "min": 1,    "max": 120,  "nulos": "No",               "valores_válidos": None},
    {"columna": "sex",       "regla": "cat",   "min": None, "max": None, "nulos": "No",               "valores_válidos": "Male, Female"},
    {"columna": "chest_pain","regla": "cat",   "min": None, "max": None, "nulos": "No",               "valores_válidos": "typical, nontypical, nonanginal, asymptomatic"},
    {"columna": "rest_bp",   "regla": "int",   "min": 50,   "max": 250,  "nulos": "No",               "valores_válidos": None},
    {"columna": "chol",      "regla": "int",   "min": 100,  "max": 600,  "nulos": "No",               "valores_válidos": None},
    {"columna": "fbs",       "regla": "bool",  "min": None, "max": None, "nulos": "No",               "valores_válidos": "True, False"},
    {"columna": "rest_ecg",  "regla": "cat",   "min": None, "max": None, "nulos": "No",               "valores_válidos": "normal, left ventricular hypertrophy, ST-T wave abnormality"},
    {"columna": "max_hr",    "regla": "int",   "min": 60,   "max": 220,  "nulos": "No",               "valores_válidos": None},
    {"columna": "exang",     "regla": "bool",  "min": None, "max": None, "nulos": "No",               "valores_válidos": "True, False"},
    {"columna": "old_peak",  "regla": "float", "min": 0.0,  "max": 10.0, "nulos": "No",               "valores_válidos": None},
    {"columna": "slope",     "regla": "cat",   "min": None, "max": None, "nulos": "No",               "valores_válidos": "1, 2, 3"},
    {"columna": "ca",        "regla": "int",   "min": 0,    "max": 3,    "nulos": "Sí (Int8 nullable)","valores_válidos": "0, 1, 2, 3"},
    {"columna": "thal",      "regla": "cat",   "min": None, "max": None, "nulos": "No",               "valores_válidos": "normal, fixed, reversable"},
    {"columna": "disease",   "regla": "bool",  "min": None, "max": None, "nulos": "No",               "valores_válidos": "True, False"},
])
validation_rules

,columna,regla,min,max,nulos,valores_válidos
0,age,int,1.0,120.0,No,NaN
1,sex,cat,NaN,NaN,No,"Male, Female"
2,chest_pain,cat,NaN,NaN,No,"typical, nontypical, nonanginal, asymptomatic"
3,rest_bp,int,50.0,250.0,No,NaN
4,chol,int,100.0,600.0,No,NaN
5,fbs,bool,NaN,NaN,No,"True, False"
6,rest_ecg,cat,NaN,NaN,No,"normal, left ventricular hypertrophy, ST-T wav..."
7,max_hr,int,60.0,220.0,No,NaN
8,exang,bool,NaN,NaN,No,"True, False"
9,old_peak,float,0.0,10.0,No,NaN


---
## 7. Modelo heurístico viable

### 7.1 Hipótesis heurística

Basándonos en el análisis bivariable (Chi-cuadrado y Mann-Whitney), las variables con mayor asociación al target `disease` son:

- **chest_pain = asymptomatic** → alta tasa de enfermedad
- **thal = reversable** → alta tasa de enfermedad  
- **exang = True** → mayor probabilidad de enfermedad
- **old_peak > 1.5** → mayor riesgo
- **ca ≥ 1** → mayor número de vasos obstruidos
- **max_hr bajo** → menos capacidad aeróbica, más riesgo

**Regla heurística:** Se predice enfermedad cardíaca si se cumplen ≥ 3 de las 6 condiciones de riesgo.

In [24]:
from sklearn.metrics import classification_report, confusion_matrix
max_hr_threshold = df["max_hr"].median()
old_peak_threshold = 1.5
min_heuristic_score = 3
def heuristica(row):
    score = 0
    if str(row["chest_pain"]) == "asymptomatic":
        score += 1
    if str(row["thal"]) == "reversable":
        score += 1
    if pd.notna(row["exang"]) and bool(row["exang"]):
        score += 1
    if pd.notna(row["old_peak"]) and row["old_peak"] > old_peak_threshold:
        score += 1
    if pd.notna(row["ca"]) and row["ca"] >= 1:
        score += 1
    if pd.notna(row["max_hr"]) and row["max_hr"] < max_hr_threshold:
        score += 1
    return score >= min_heuristic_score
df["pred_heuristica"] = df.apply(heuristica, axis=1)
y_true = df["disease"].astype(object).astype(bool)
y_pred = df["pred_heuristica"]
print("=== Reporte del modelo heurístico ===")
print(classification_report(y_true, y_pred,
      target_names=["Sin enfermedad", "Con enfermedad"]))

=== Reporte del modelo heurístico ===


                precision    recall  f1-score   support

Sin enfermedad       0.80      0.92      0.86      1569
Con enfermedad       0.88      0.73      0.80      1295

      accuracy                           0.83      2864
     macro avg       0.84      0.82      0.83      2864
  weighted avg       0.84      0.83      0.83      2864



In [25]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: No", "Pred: Sí"],
            yticklabels=["Real: No", "Real: Sí"], ax=ax)
ax.set_title("Matriz de confusión — Heurística")
plt.tight_layout()
plt.show()

/tmp/ipykernel_932/4136668770.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8. Conclusiones e identificación de transformaciones

### 8.1 Hallazgos principales

#### Variables más discriminantes para predecir enfermedad cardíaca:

**Muy alta asociación (p < 0.001):**
- `chest_pain` (asymptomatic → alto riesgo), `thal` (reversable → alto riesgo),  
  `ca`, `exang`, `old_peak`, `max_hr`, `slope`

**Asociación moderada:**
- `age`, `rest_bp`, `sex`

**Baja asociación:**
- `fbs`, `chol`, `rest_ecg`

#### Distribuciones:
- Ninguna variable numérica sigue distribución normal (Shapiro-Wilk, p < 0.05).  
- `old_peak` y `chol` presentan skewness positivo, candidatos a transformación log.
- `rest_bp` y `chol` tienen outliers significativos (valores fisiológicamente posibles pero extremos).

#### Nulos:
- `ca` es la única variable con valores nulos (heredados del dataset original).

### 8.2 Transformaciones recomendadas para ML

| Transformación | Variables | Justificación |
|---|---|---|
| Imputación de medianas | `ca` | Nulos presentes |
| Escalado (StandardScaler / RobustScaler) | age, rest_bp, chol, max_hr, old_peak | Modelos sensibles a escala (SVM, KNN, regresión logística) |
| Log transform | old_peak, chol | Skewness positivo alto |
| One-Hot Encoding | sex, chest_pain, rest_ecg, thal | Nominales sin orden |
| Ordinal Encoding | slope | Ordinal 1 < 2 < 3 |
| Manejo de outliers | rest_bp, chol | Capping en percentil 99 |

### 8.3 ¿Es viable un modelo heurístico?

**Sí, es viable como baseline.** El modelo heurístico simple (≥ 3 factores de riesgo) logra una accuracy razonable, con buena recall para la clase positiva (enfermos), que es la métrica clínica más importante para minimizar falsos negativos (pacientes enfermos no detectados).

Sin embargo, dado el tamaño del dataset (~3000 muestras limpias) y la cantidad de variables informativas, **se recomienda pasar a un modelo supervisado** (árboles de decisión, Random Forest, XGBoost o regresión logística) que podrá capturar interacciones no lineales entre variables y superar al heurístico en F1-score.

### 8.4 Datos adicionales que serían útiles
- Biomarcadores adicionales: troponina, BNP (péptido natriurético)
- Historial familiar de enfermedades cardíacas
- Medicación actual del paciente
- Índice de masa corporal (IMC)
- Electrocardiograma completo (no solo rest_ecg resumido)

In [26]:
# Limpieza: eliminar columnas temporales
df.drop(columns=["pred_heuristica", "disease_str"], inplace=True)
print("Notebook completado correctamente.")

Notebook completado correctamente.
